# Chapter 15 — Exercises: LLMs and Privacy

This notebook contains starter code and data for the exercises in
Lecture 15 of *Large Language Models in Finance*.

See `course/lectures/15-privacy-local-models/exercises.md` for the full exercise statements
and `solutions.md` for the worked solutions.

## Data Lab -- SEC EDGAR

Scan 10-K filings for PII, implement heuristic NER, and build a redaction pipeline.

In [ ]:
import requests, time, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

EDGAR_HEADERS = {"User-Agent": "LLM-Finance-Course instructor@dauphine.eu"}

def edgar_get(url):
    time.sleep(0.11)
    r = requests.get(url, headers=EDGAR_HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()

def get_cik(ticker):
    data = edgar_get("https://www.sec.gov/files/company_tickers.json")
    for v in data.values():
        if v["ticker"].upper() == ticker.upper():
            return str(v["cik_str"]).zfill(10)
    raise ValueError(f"Ticker {ticker} not found")

def get_submissions(cik):
    return edgar_get(f"https://data.sec.gov/submissions/CIK{cik}.json")

def get_concept(cik, concept):
    return edgar_get(f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{concept}.json")

def get_annual_series(cik, concept):
    data = get_concept(cik, concept)
    usd  = data.get("units", {}).get("USD", [])
    rows = [u for u in usd if u.get("form") == "10-K" and u.get("filed")]
    if not rows:
        for _, vals in data.get("units", {}).items():
            rows = [u for u in vals if u.get("form") == "10-K" and u.get("filed")]
            if rows: break
    df = (pd.DataFrame(rows)[["end","val","filed","accn"]]
            .rename(columns={"end":"date","val":concept})
            .drop_duplicates("date").sort_values("date"))
    df["date"] = pd.to_datetime(df["date"])
    return df

def fetch_10k_html(ticker):
    cik  = get_cik(ticker)
    subs = get_submissions(cik)
    f    = subs["filings"]["recent"]
    idx  = next(i for i, x in enumerate(f["form"]) if x == "10-K")
    acc  = f["accessionNumber"][idx].replace("-", "")
    url  = (f"https://www.sec.gov/Archives/edgar/data/{cik.lstrip('0')}"
            f"/{acc}/{f['primaryDocument'][idx]}")
    time.sleep(0.15)
    return requests.get(url, headers=EDGAR_HEADERS, timeout=30).text

def extract_mda(html):
    text = re.sub(r"<[^>]+>", " ", html)
    m    = re.search(
        r"Item\s+7[.\s]+Management.{0,80}Discussion.*?(?=Item\s+7A|Item\s+8)",
        text, re.IGNORECASE | re.DOTALL)
    raw  = m.group(0) if m else text[:30000]
    return re.sub(r"\s+", " ", raw).strip()

### Exercise [B]: Regex PII Scanner

In [ ]:
# --- Exercise [B]: PII Scanner ---
html_aapl15 = fetch_10k_html("AAPL")
text_full   = re.sub(r"<[^>]+>", " ", html_aapl15)
text_full   = re.sub(r"\s+", " ", text_full)

PII_PATTERNS = {
    "SSN":         r"\b\d{3}-\d{2}-\d{4}\b",
    "Email":       r"\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Z|a-z]{2,}\b",
    "Phone":       r"\b(?:\+?1[-.\s]?)?\(?\d{3}\)?[-.\s]\d{3}[-.\s]\d{4}\b",
    "DOB":         r"\b(?:born|DOB|date of birth).{0,30}\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}",
}

total_counts = {}
for ptype, pat in PII_PATTERNS.items():
    matches = list(re.finditer(pat, text_full, re.IGNORECASE))
    total_counts[ptype] = len(matches)
    for m in matches[:3]:
        start = max(0, m.start()-20); end = min(len(text_full), m.end()+20)
        print(f"[{ptype}] ...{text_full[start:end]}...")
    if not matches: print(f"[{ptype}] No matches found (expected for 10-K)")

print("\nPII match counts:")
for k, v in total_counts.items():
    print(f"  {k:<10}: {v}")

### Exercise [I]: Heuristic NER

In [ ]:
# --- Exercise [I]: Heuristic NER ---
text_full15 = re.sub(r"<[^>]+>", " ", html_aapl15)
risk_m = re.search(r"Item\s+1A.*?(?=Item\s+1B|Item\s+2)", text_full15, re.IGNORECASE | re.DOTALL)
risk_text15 = re.sub(r"\s+", " ", risk_m.group(0)) if risk_m else text_full15[:100000]

# PERSON: title followed by capitalised name
person_pat = r"(?:Mr\.|Ms\.|Dr\.|President|CEO|CFO|Chairman|Chief\s+\w+)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,2})"
persons = re.findall(person_pat, risk_text15)

# ORG: tokens/bigrams ending in corporate suffixes
org_pat = r"[A-Z][A-Za-z\s]{2,40}(?:Inc\.|Corp\.|LLC|Ltd\.|Corporation|Company|Group|Holdings)"
orgs    = re.findall(org_pat, risk_text15)

from collections import Counter
person_counts = Counter(persons)
org_counts    = Counter(o.strip() for o in orgs)

entities = (
    [(name, "PERSON", cnt) for name, cnt in person_counts.most_common(10)] +
    [(name, "ORG",    cnt) for name, cnt in org_counts.most_common(10)]
)
entities.sort(key=lambda x: -x[2])

print(f"{'Entity':<45} {'Type':<8} {'Count'}")
print("-"*65)
for ent, etype, cnt in entities[:20]:
    print(f"{ent[:44]:<45} {etype:<8} {cnt}")

### Exercise [A]: Redaction Pipeline

In [ ]:
# --- Exercise [A]: Redaction Pipeline ---
words_5k = " ".join(text_full15.split()[:5000])

def redact(text, persons):
    out = text
    for pat in PII_PATTERNS.values():
        out = re.sub(pat, "[REDACTED]", out, flags=re.IGNORECASE)
    for name in persons:
        if len(name) > 4:
            out = re.sub(re.escape(name), "[PERSON]", out)
    return out

redacted = redact(words_5k, list(person_counts.keys()))

print("=== Redacted sample (first 1000 chars) ===")
print(redacted[:1000])

# Utility score
orig_sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", words_5k) if len(s.split()) >= 3]
red_sents  = [s.strip() for s in re.split(r"(?<=[.!?])\s+", redacted) if len(s.split()) >= 3]

useful = 0
for rs in red_sents:
    non_redacted = [w for w in rs.split() if w not in {"[REDACTED]","[PERSON]"}]
    if len(non_redacted) >= 2: useful += 1

utility = useful / max(len(red_sents), 1)
redact_density = (redacted.count("[REDACTED]") + redacted.count("[PERSON]")) / max(len(words_5k.split()),1)
print(f"\nUtility score:      {utility:.3f}  ({useful}/{len(red_sents)} sentences usable)")
print(f"Redaction density:  {redact_density:.4f} redactions per word")